# 📄 Resume Score Checker ML - Training Notebook

This notebook provides a complete walk-through to train the machine-learning-powered **Resume Scorer engine**. It extracts structured NLP features from resumes, trains a calibrated **XGBoost Regressor**, and utilizes **SHAP (SHapley Additive exPlanations)** to inspect and explain the features driving the score.

## 🛠️ Step 1: Environment Setup
First, we will clone the repository, navigate to the backend workspace, and install the necessary dependencies including `spacy` model files.

In [ ]:
# 1. Clone the GitHub Repository
!git clone https://github.com/sreeram0343/resume-score-ml.git
%cd resume-score-ml/backend

In [ ]:
# 2. Install Project Dependencies
!pip install -r requirements.txt

# 3. Download spaCy English Language Model
!python -m spacy download en_core_web_sm

## 📂 Step 2: Labeled Data Preparation
To train the model, we need a dataset containing:
- `resume_text`: Full parsed text of candidate resumes.
- `job_description`: Job description being evaluated against.
- `target_role`: The title of the target position.
- `ats_score`: A target numeric score (0 to 100) indicating the resume match rating.

If you don't have a dataset ready, the script below will generate a mock dataset with 100 rows containing variations of software engineer, frontend dev, data scientist, and project manager roles to demonstrate the pipeline.

In [ ]:
import os
import pandas as pd
import numpy as np

os.makedirs("./data", exist_ok=True)
dataset_path = "./data/dataset.csv"

if not os.path.exists(dataset_path):
    print("Creating dummy dataset for training demonstration...")
    resumes = [
        "Python developer with strong FastAPI, PostgreSQL, Docker, and Kubernetes skills. Experienced building microservices.",
        "Frontend engineer specializing in React, Next.js, TypeScript, HTML5 and Tailwind CSS. Built responsive web applications.",
        "Data scientist with hands-on experience in pandas, numpy, scikit-learn, XGBoost, and SHAP. Machine learning engineering expert.",
        "Senior project manager and Agile coach with CSM. Led cross-functional Scrum teams and managed product backlogs."
    ] * 25
    
    jds = [
        "Seeking a Backend Engineer proficient in Python, FastAPI, Postgres, and Docker. Experience with microservices is required.",
        "We are hiring a Frontend Developer experienced in building modern web apps using React, Next.js, and CSS frameworks.",
        "Looking for an ML Engineer with python expertise, strong understanding of pandas, scikit-learn, and boosting models.",
        "Hiring an Agile Project Manager to facilitate scrum events, sprint planning, and coordinate deliverable timelines."
    ] * 25
    
    roles = [
        "Backend Engineer",
        "Frontend Engineer",
        "Machine Learning Engineer",
        "Project Manager"
    ] * 25
    
    # Generating scores with realistic variations depending on role alignment
    ats_scores = np.random.uniform(70, 95, 100)
    
    df = pd.DataFrame({
        "resume_text": resumes,
        "job_description": jds,
        "target_role": roles,
        "ats_score": ats_scores
    })
    df.to_csv(dataset_path, index=False)
    print(f"Dummy dataset successfully saved to {dataset_path} ({len(df)} samples).")
else:
    print(f"Dataset already exists at {dataset_path}.")

## 🔬 Step 3: Run Feature Extraction & XGBoost Scorer Training
Let's execute the training CLI module. This will:
1. Load the CSV file and iterate over the entries.
2. Parse segments (Skills, Experience, Education) using our spaCy NLP extraction layers.
3. Compute multi-stage features (ATS compliance, Keyword matching, Content quality, Semantic similarity).
4. Train an **XGBoost regressor** model with 5-fold cross-validation, and perform isotonic score calibration.
5. Save the trained pipeline (`models/scorer.pkl`).

In [ ]:
# Run the ML Pipeline Training
!python -m ml_pipeline.train --dataset_path ./data/dataset.csv --output_path ./models/scorer.pkl

## 📊 Step 4: Interactive Training and Evaluation Plots
Now let's load the classes interactively in the notebook, train them, and visualize performance using inline plots.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Add current directory to path to enable local module imports
sys.path.append(os.getcwd())

from app.parsers.dispatcher import ParserDispatcher
from app.parsers.section_extractor import SectionExtractor
from app.features.ats_features import ATSFeatureExtractor
from app.features.content_features import ContentFeatureExtractor
from app.features.keyword_features import KeywordFeatureExtractor
from app.features.semantic_features import SemanticFeatureExtractor
from app.models.feature_assembler import FeatureAssembler
from app.models.xgboost_scorer import ResumeScorer
from app.parsers.schemas import ParseResult, ATSFlags

In [ ]:
# 1. Read Data
df = pd.read_csv("./data/dataset.csv")
X = []
y = []

print("Starting interactive feature extraction...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    resume_text = row["resume_text"]
    jd = row["job_description"]
    role = row.get("target_role", "Software Engineer")
    
    # Parse details
    pr = ParseResult(
        text=resume_text,
        word_count=len(resume_text.split()),
        page_count=1,
        ats_flags=ATSFlags(),
        parser_used="txt"
    )
    
    rd = SectionExtractor.extract(resume_text)
    
    # Compute feature vector metrics
    ats_f = ATSFeatureExtractor.extract(pr, rd)
    content_f = ContentFeatureExtractor.extract(rd)
    keyword_f = KeywordFeatureExtractor.extract(resume_text, role, job_description=jd)
    semantic_f = SemanticFeatureExtractor.extract(
        resume_text, role, job_description=jd,
        skills_text=", ".join(rd.skills),
        experience_text="\n".join([f"{e.company} {e.title}" for e in rd.experience])
    )
    
    feat_vec = FeatureAssembler.assemble(ats_f, content_f, keyword_f, semantic_f)
    X.append(feat_vec)
    y.append(row["ats_score"])

X = np.array(X)
y = np.array(y)
print(f"Feature matrix compiled successfully: {X.shape}")

In [ ]:
# 2. Train and Validate
scorer = ResumeScorer()
result = scorer.train(X, y, FeatureAssembler.get_feature_names())

print("\n--- Model Performance Summary ---")
print(f"Cross-Validation Mean MAE: {np.mean(result.cv_scores):.4f}")
print(f"Validation Mean Absolute Error (MAE): {result.mae:.4f}")
print(f"Validation Root Mean Squared Error (RMSE): {result.rmse:.4f}")
print(f"R2 score (Coefficient of Determination): {result.r2:.4f}")

In [ ]:
# 3. Plot Actual vs. Predicted Scores
y_pred = [scorer.predict(x) for x in X]
plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.6, color='#2563EB', edgecolors='black', label="Predicted samples")
plt.plot([min(y)-5, max(y)+5], [min(y)-5, max(y)+5], 'r--', linewidth=2, label="Perfect Alignment")
plt.xlabel("Actual Target Resume Score")
plt.ylabel("Predicted Model Resume Score")
plt.title("Scoring Alignment: Actual vs. Predicted Values")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

In [ ]:
# 4. Plot Feature Importances derived from XGBoost Gains
plt.figure(figsize=(12, 7))
importances = result.feature_importances
names = list(importances.keys())[:15]
vals = [importances[n] for n in names]

plt.barh(names, vals, color='#7C3AED', edgecolor='black')
plt.gca().invert_yaxis()
plt.xlabel("Normalized Relative Gain (Importance)")
plt.title("Top 15 Feature Importances Driving the Regressor")
plt.grid(True, axis='x', linestyle='--', alpha=0.4)
plt.show()

## 🔮 Step 5: Explaining Predictions using SHAP
We can leverage SHAP values to explain *why* the model assigned a specific score to a resume. This extracts positive factors contributing to the score, and negative factors pulling it down.

In [ ]:
from app.models.explainer import ResumeSHAPExplainer
import shap

# Initialize Explainer
explainer = ResumeSHAPExplainer()
explainer.initialize(scorer.model, scorer.feature_names)

# Let's explain the first resume in our dataset
sample_idx = 0
sample_vector = X[sample_idx]
score = scorer.predict(sample_vector)
target_role = df.loc[sample_idx, "target_role"]

explanation = explainer.explain(sample_vector, score, role=target_role, has_jd=True)

print(f"\n🔍 [Evaluation Outcome]")
print(f"Target Role: {target_role}")
print(f"Overall Score: {score:.1f}/100")
print(f"Human Explanation: {explanation.explanation_text}\n")

print("🟢 Positive Contributors:")
for f in explanation.positive_factors:
    print(f"  - {f.human_label} ({f.feature_name}): {f.impact_label}")

print("\n🔴 Negative Contributors:")
for f in explanation.negative_factors:
    print(f"  - {f.human_label} ({f.feature_name}): {f.impact_label}")

In [ ]:
# Visualizing SHAP summary plots over the dataset
scaled_X = scorer.scaler.transform(X)
shap_values = explainer.explainer.shap_values(scaled_X)

plt.title("SHAP Force/Influence Summary Plot")
shap.summary_plot(shap_values, scaled_X, feature_names=scorer.feature_names)

---